# Spread and Z-Score Strategy — Pairs Trading

This notebook constructs the spread between a cointegrated pair of stocks
using a hedge ratio estimated via linear regression.

The spread is standardized into a z-score using a rolling window.  
Trading signals are generated when the z-score deviates significantly
from its mean, indicating a potential mean-reversion opportunity.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Load Cleaned Price Data

We load the cleaned NIFTY 50 closing price dataset generated
during the preprocessing stage.


In [ ]:
close_prices = pd.read_csv(
    "../data/processed/nifty50_close_prices.csv",
    index_col="Date",
    parse_dates=True
)

## Select Cointegrated Pair

We select one cointegrated pair identified in the previous notebook
to demonstrate the spread construction and signal generation process.


In [ ]:
stock1 = close_prices["EICHERMOT.NS"]
stock2 = close_prices["MARUTI.NS"]

## Estimate Hedge Ratio

We estimate the hedge ratio using Ordinary Least Squares (OLS).

Stock1 = α + β × Stock2

The coefficient β represents the hedge ratio used to construct the spread.


In [ ]:
import statsmodels.api as sm

# add constant for regression
X = sm.add_constant(stock2)

# run OLS regression to estimate hedge ratio
model = sm.OLS(stock1, X).fit()

# hedge ratio
beta = model.params[stock2.name]

print("Beta:", beta)

## Construct Spread

The spread represents the deviation between the two assets
after adjusting for the hedge ratio.

Spread = P₁ − β × P₂


In [ ]:
spread = stock1 - beta*stock2

## Z-Score Calculation

The spread is standardized using a rolling window.

Z = (Spread − Rolling Mean) / Rolling Std

Extreme values of the z-score indicate potential trading opportunities.


In [ ]:
window = 60 # rolling window length

rolling_mean = spread.rolling(window).mean()
rolling_std = spread.rolling(window).std()

# standardized spread
z_score = (spread - rolling_mean) / rolling_std

## Trading Signal Generation

Trading signals are generated based on z-score thresholds.

Long Entry  : z < −2  
Short Entry : z > 2  
Exit        : |z| < 0.2


In [ ]:
signals = pd.DataFrame(index=z_score.index)

signals["z_score"] = z_score

# entry signals
signals["long"] = z_score < -2
signals["short"] = z_score > 2

# exit condition
signals["exit"] = abs(z_score) < 0.2

signals.head()
print("Total long signals:", signals["long"].sum())
print("Total short signals:", signals["short"].sum())


## Save Trading Signals

The generated trading signals are saved for use in the
backtesting stage of the strategy.


In [ ]:
signals.to_csv("../data/processed/trading_signals.csv")

print("Total signals:", len(signals))


print("Trading signals saved successfully")


## Visualization of Trading Signals

The plot below shows the z-score of the spread along with
long and short trading signals based on threshold levels.


In [ ]:
plt.figure(figsize=(12,6))

plt.plot(z_score, label="Z Score")

plt.axhline(2, color="red", linestyle="--")
plt.axhline(-2, color="green", linestyle="--")
plt.axhline(0, color="black")

plt.scatter(z_score[z_score > 2].index,
            z_score[z_score > 2],
            color="red",
            label="Short Signal")

plt.scatter(z_score[z_score < -2].index,
            z_score[z_score < -2],
            color="green",
            label="Long Signal")

plt.legend()
plt.title("Trading Signals")
plt.savefig("../reports/zscore_trading_signal.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(spread)
plt.title("Spread between EICHERMOT and MARUTI")
plt.savefig("../reports/spread_series.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(z_score)

plt.axhline(2, color="red")
plt.axhline(-2, color="green")
plt.axhline(0, color="black")

plt.title("Z-Score")
plt.savefig("../reports/zscore_trading_zones.png", dpi=300, bbox_inches="tight")
plt.show()

## Output

This notebook produces:

• Spread between the selected cointegrated pair  
• Rolling z-score of the spread  
• Long / short / exit trading signals  
• Visualization of spread and z-score behaviour

These signals will be used in the next notebook for strategy backtesting.
